# 00 - Colab setup

Run this notebook once per Colab session (Runtime > Run all). It:

1. mounts Google Drive (all data, adapters and results live there),
2. clones or updates the private GitHub repo into `/content/fedcti-llm-v3`,
3. installs the requirements and the `fedcti` package,
4. sets git identity so any commit made from Colab is yours,
5. checks the GPU and writes a smoke-test record to Drive.

One-time prerequisite: add a GitHub token as a Colab secret named `GITHUB_TOKEN`
(key icon in the left sidebar, enable notebook access). See `docs/colab_workflow.md`.

In [ ]:
GITHUB_USER = "YOUR_GITHUB_USERNAME"
REPO_NAME = "fedcti-llm-v3"
BRANCH = "main"
GIT_NAME = "Rishabh Khale"
GIT_EMAIL = "khalerishabhproff@gmail.com"
REPO_DIR = f"/content/{REPO_NAME}"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")
url = f"https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git"

if os.path.isdir(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", url], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, url, REPO_DIR], check=True)

subprocess.run(["git", "-C", REPO_DIR, "config", "user.name", GIT_NAME], check=True)
subprocess.run(["git", "-C", REPO_DIR, "config", "user.email", GIT_EMAIL], check=True)
print(subprocess.check_output(["git", "-C", REPO_DIR, "log", "--oneline", "-3"], text=True))

In [ ]:
%pip install -q -r {REPO_DIR}/requirements.txt
%pip install -q -e {REPO_DIR}

In [ ]:
# A fresh editable install is only importable after the path is refreshed.
import sys, importlib, site
importlib.reload(site)
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

import torch
from fedcti.config import load_config, storage_root
from fedcti.repro import save_metadata

cfg = load_config("exp2/data.yaml")
root = storage_root(cfg)
print("storage root:", root)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("bf16 supported:", torch.cuda.is_available() and torch.cuda.is_bf16_supported())
print(save_metadata(root / "logs" / "setup_smoke.json", cfg))